In [2]:
import pandas as pd

# 1. Inspect the agents picked/banned
df_draft = pd.read_csv("data/vct_2024/matches/draft_phase.csv")
print("=== DRAFT PHASE ===")
display(df_draft.head())

# 2. Inspect the match outcomes and map names
df_overview = pd.read_csv("data/vct_2024/matches/overview.csv")
print("\n=== OVERVIEW ===")
display(df_overview.head())

=== DRAFT PHASE ===


,Tournament,Stage,Match Type,Match Name,Team,Action,Map
0,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Sentinels,ban,Icebox
1,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Gen.G,ban,Sunset
2,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Sentinels,pick,Haven
3,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Gen.G,pick,Ascent
4,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Sentinels,ban,Bind



=== OVERVIEW ===


,Tournament,Stage,Match Type,Match Name,Map,Player,Team,Agents,Rating,Average Combat Score,...,Deaths,Assists,Kills - Deaths (KD),"Kill, Assist, Trade, Survive %",Average Damage Per Round,Headshot %,First Kills,First Deaths,Kills - Deaths (FKD),Side
0,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,t3xture,Gen.G,jett,1.60,305.0,...,12.0,4.0,12.0,81%,188.0,31%,3.0,4.0,-1.0,both
1,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,t3xture,Gen.G,jett,1.71,364.0,...,5.0,2.0,7.0,89%,216.0,37%,2.0,2.0,0.0,attack
2,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,t3xture,Gen.G,jett,1.52,261.0,...,7.0,2.0,5.0,75%,168.0,24%,1.0,2.0,-1.0,defend
3,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,Meteor,Gen.G,killjoy,1.23,221.0,...,10.0,4.0,7.0,76%,140.0,26%,1.0,4.0,-3.0,both
4,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,Meteor,Gen.G,killjoy,1.16,229.0,...,4.0,1.0,4.0,78%,127.0,35%,1.0,1.0,0.0,attack


In [3]:
# 1. Inspect maps_scores.csv to find the winner
df_scores = pd.read_csv("data/vct_2024/matches/maps_scores.csv")
print("=== MAP SCORES ===")
display(df_scores.head())

# 2. Group the 5 agents per team from overview.csv
# Filter 'both' side to avoid duplicate agent names for the same player
df_agents = df_overview[df_overview['Side'] == 'both']

# Group by Match, Map, and Team to get a list of 5 agents
df_team_comps = df_agents.groupby(['Match Name', 'Map', 'Team'])['Agents'].apply(list).reset_index()
print("\n=== AGENT COMPOSITIONS ===")
display(df_team_comps.head(6))

=== MAP SCORES ===


,Tournament,Stage,Match Type,Match Name,Map,Team A,Team A Score,Team A Attacker Score,Team A Defender Score,Team A Overtime Score,Team B,Team B Score,Team B Attacker Score,Team B Defender Score,Team B Overtime Score,Duration
0,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Haven,Gen.G,13,7,6,NaN,Sentinels,8,3,5,NaN,1:02:40
1,Valorant Champions 2024,Group Stage,Opening (B),Gen.G vs Sentinels,Ascent,Gen.G,13,8,5,NaN,Sentinels,7,3,4,NaN,46:45
2,Valorant Champions 2024,Group Stage,Opening (B),FunPlus Phoenix vs Team Heretics,Abyss,FunPlus Phoenix,12,8,4,0.0,Team Heretics,14,8,4,2.0,1:10:12
3,Valorant Champions 2024,Group Stage,Opening (B),FunPlus Phoenix vs Team Heretics,Bind,FunPlus Phoenix,13,8,5,NaN,Team Heretics,9,5,4,NaN,53:01
4,Valorant Champions 2024,Group Stage,Opening (B),FunPlus Phoenix vs Team Heretics,Lotus,FunPlus Phoenix,13,6,6,1.0,Team Heretics,15,6,6,3.0,1:07:12



=== AGENT COMPOSITIONS ===


,Match Name,Map,Team,Agents
0,100 Thieves vs FUT Esports,All Maps,100 Thieves,"[brimstone, jett, fade, sova, gekko, killjoy, ..."
1,100 Thieves vs FUT Esports,All Maps,FUT Esports,"[omen, viper, jett, raze, killjoy, viper, kayo..."
2,100 Thieves vs FUT Esports,Bind,100 Thieves,"[brimstone, gekko, raze, fade, viper]"
3,100 Thieves vs FUT Esports,Bind,FUT Esports,"[viper, raze, omen, kayo, skye]"
4,100 Thieves vs FUT Esports,Icebox,100 Thieves,"[sova, viper, jett, killjoy, gekko]"
5,100 Thieves vs FUT Esports,Icebox,FUT Esports,"[viper, jett, sova, kayo, killjoy]"


In [4]:
# 1. Matikan limit lebar kolom agar teks tidak dipotong
pd.set_option('display.max_colwidth', None)

# 2. Buang baris yang Map-nya berisi "All Maps"
df_team_comps = df_team_comps[df_team_comps['Map'] != 'All Maps']

print("=== CLEANED AGENT COMPOSITIONS ===")
display(df_team_comps.head(4))

# 3. Buat kolom penentu pemenang (Winner) di tabel skor
# Jika skor A > skor B, maka Team A yang menang, dan sebaliknya
df_scores['Winner'] = df_scores.apply(
    lambda row: row['Team A'] if row['Team A Score'] > row['Team B Score'] else row['Team B'], 
    axis=1
)

print("\n=== SCORES WITH WINNER LABEL ===")
display(df_scores[['Match Name', 'Map', 'Team A', 'Team B', 'Winner']].head())

=== CLEANED AGENT COMPOSITIONS ===


,Match Name,Map,Team,Agents
2,100 Thieves vs FUT Esports,Bind,100 Thieves,"[brimstone, gekko, raze, fade, viper]"
3,100 Thieves vs FUT Esports,Bind,FUT Esports,"[viper, raze, omen, kayo, skye]"
4,100 Thieves vs FUT Esports,Icebox,100 Thieves,"[sova, viper, jett, killjoy, gekko]"
5,100 Thieves vs FUT Esports,Icebox,FUT Esports,"[viper, jett, sova, kayo, killjoy]"



=== SCORES WITH WINNER LABEL ===


,Match Name,Map,Team A,Team B,Winner
0,Gen.G vs Sentinels,Haven,Gen.G,Sentinels,Gen.G
1,Gen.G vs Sentinels,Ascent,Gen.G,Sentinels,Gen.G
2,FunPlus Phoenix vs Team Heretics,Abyss,FunPlus Phoenix,Team Heretics,Team Heretics
3,FunPlus Phoenix vs Team Heretics,Bind,FunPlus Phoenix,Team Heretics,FunPlus Phoenix
4,FunPlus Phoenix vs Team Heretics,Lotus,FunPlus Phoenix,Team Heretics,Team Heretics


In [5]:
# 1. First merge: Retrieve agent composition for Team A
df_final = pd.merge(
    df_scores[['Match Name', 'Map', 'Team A', 'Team B', 'Winner']], 
    df_team_comps, 
    left_on=['Match Name', 'Map', 'Team A'], 
    right_on=['Match Name', 'Map', 'Team'], 
    how='inner'
).rename(columns={'Agents': 'Team_A_Agents'}).drop(columns=['Team'])

# 2. Second merge: Retrieve agent composition for Team B
df_final = pd.merge(
    df_final, 
    df_team_comps, 
    left_on=['Match Name', 'Map', 'Team B'], 
    right_on=['Match Name', 'Map', 'Team'], 
    how='inner'
).rename(columns={'Agents': 'Team_B_Agents'}).drop(columns=['Team'])

# 3. Convert Target to Binary for XGBoost (1 = Team A Win, 0 = Team A Loss)
df_final['Target_Team_A_Win'] = (df_final['Winner'] == df_final['Team A']).astype(int)

# Display the final structure of our Machine Learning dataset
print("=== FINAL ML DATASET STRUCTURE ===")
display(df_final[['Map', 'Team_A_Agents', 'Team_B_Agents', 'Target_Team_A_Win']].head())

=== FINAL ML DATASET STRUCTURE ===


,Map,Team_A_Agents,Team_B_Agents,Target_Team_A_Win
0,Haven,"[jett, killjoy, skye, omen, viper]","[sova, omen, breach, cypher, jett]",1
1,Ascent,"[omen, kayo, killjoy, jett, sova, kayo, sova, jett, killjoy, omen]","[sova, deadlock, cypher, jett, omen, omen, sova, jett, killjoy, kayo]",1
2,Abyss,"[fade, viper, omen, cypher, yoru]","[sova, gekko, cypher, astra, jett]",0
3,Bind,"[chamber, viper, yoru, fade, omen]","[viper, gekko, brimstone, fade, raze]",1
4,Lotus,"[viper, astra, gekko, omen, raze, raze, astra, gekko, viper, omen]","[viper, raze, fade, omen, killjoy, raze, cypher, breach, omen, fade]",0


In [6]:
# Remove duplicate agents by converting lists to sets, then back to lists
df_final['Team_A_Agents'] = df_final['Team_A_Agents'].apply(lambda x: list(set(x)))
df_final['Team_B_Agents'] = df_final['Team_B_Agents'].apply(lambda x: list(set(x)))

# Verify that the anomalies (like index 1 and 4) are now fixed to exactly 5 agents
print("=== CLEANED FINAL DATASET ===")
display(df_final[['Map', 'Team_A_Agents', 'Team_B_Agents', 'Target_Team_A_Win']].head())

=== CLEANED FINAL DATASET ===


,Map,Team_A_Agents,Team_B_Agents,Target_Team_A_Win
0,Haven,"[jett, viper, omen, skye, killjoy]","[jett, omen, sova, cypher, breach]",1
1,Ascent,"[jett, omen, kayo, killjoy, sova]","[jett, deadlock, omen, kayo, sova, cypher, killjoy]",1
2,Abyss,"[fade, viper, omen, cypher, yoru]","[jett, gekko, sova, cypher, astra]",0
3,Bind,"[fade, chamber, viper, omen, yoru]","[fade, raze, viper, gekko, brimstone]",1
4,Lotus,"[raze, viper, omen, gekko, astra]","[fade, raze, viper, omen, killjoy, cypher, breach]",0


In [7]:
# 1. Define a stricter composite key to isolate exact matches
composite_keys = ['Tournament', 'Stage', 'Match Name', 'Map']

# 2. Regroup the agents using the stricter keys
df_team_comps_strict = df_agents.groupby(composite_keys + ['Team'])['Agents'].apply(list).reset_index()
df_team_comps_strict = df_team_comps_strict[df_team_comps_strict['Map'] != 'All Maps']

# Clean duplicates within the same map and strictly filter to exact 5v5 compositions
df_team_comps_strict['Agents'] = df_team_comps_strict['Agents'].apply(lambda x: list(set(x)))
df_team_comps_strict = df_team_comps_strict[df_team_comps_strict['Agents'].apply(len) == 5]

# 3. Redo the first merge for Team A
df_final = pd.merge(
    df_scores[composite_keys + ['Team A', 'Team B', 'Winner']], 
    df_team_comps_strict, 
    left_on=composite_keys + ['Team A'], 
    right_on=composite_keys + ['Team'], 
    how='inner'
).rename(columns={'Agents': 'Team_A_Agents'}).drop(columns=['Team'])

# 4. Redo the second merge for Team B
df_final = pd.merge(
    df_final, 
    df_team_comps_strict, 
    left_on=composite_keys + ['Team B'], 
    right_on=composite_keys + ['Team'], 
    how='inner'
).rename(columns={'Agents': 'Team_B_Agents'}).drop(columns=['Team'])

# 5. Apply the binary target variable
df_final['Target_Team_A_Win'] = (df_final['Winner'] == df_final['Team A']).astype(int)

print("=== STRICT CLEANED FINAL DATASET ===")
display(df_final[['Tournament', 'Map', 'Team_A_Agents', 'Team_B_Agents', 'Target_Team_A_Win']].head())

=== STRICT CLEANED FINAL DATASET ===


,Tournament,Map,Team_A_Agents,Team_B_Agents,Target_Team_A_Win
0,Valorant Champions 2024,Haven,"[jett, viper, omen, skye, killjoy]","[jett, omen, sova, cypher, breach]",1
1,Valorant Champions 2024,Ascent,"[jett, omen, kayo, killjoy, sova]","[jett, deadlock, omen, sova, cypher]",1
2,Valorant Champions 2024,Abyss,"[fade, viper, omen, cypher, yoru]","[jett, gekko, sova, cypher, astra]",0
3,Valorant Champions 2024,Bind,"[fade, chamber, viper, omen, yoru]","[fade, raze, viper, gekko, brimstone]",1
4,Valorant Champions 2024,Lotus,"[raze, viper, omen, gekko, astra]","[fade, raze, viper, omen, killjoy]",0


In [10]:
!pip install scikit-learn xgboost

   ---------------------------------------- 0.0/125.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/125.4 MB 5.9 MB/s eta 0:00:22
   ---------------------------------------- 0.8/125.4 MB 8.7 MB/s eta 0:00:15
   ---------------------------------------- 1.4/125.4 MB 9.7 MB/s eta 0:00:13
    --------------------------------------- 1.9/125.4 MB 10.1 MB/s eta 0:00:13
    --------------------------------------- 2.1/125.4 MB 10.2 MB/s eta 0:00:13
    --------------------------------------- 2.1/125.4 MB 10.2 MB/s eta 0:00:13
    --------------------------------------- 2.1/125.4 MB 10.2 MB/s eta 0:00:13
    --------------------------------------- 2.3/125.4 MB 6.2 MB/s eta 0:00:20
    --------------------------------------- 2.9/125.4 MB 6.9 MB/s eta 0:00:18
   - -------------------------------------- 3.5/125.4 MB 7.3 MB/s eta 0:00:17
   - -------------------------------------- 4.0/125.4 MB 7.7 MB/s eta 0:00:16
   - -------------------------------------- 4.5/125.4 MB 8.0 MB/s e

In [15]:
%pip install scikit-learn xgboost

                                              0.0/8.3 MB ? eta -:--:--
                                              0.2/8.3 MB 6.3 MB/s eta 0:00:02
     ---                                      0.7/8.3 MB 8.5 MB/s eta 0:00:01
     -----                                    1.2/8.3 MB 9.6 MB/s eta 0:00:01
     --------                                 1.7/8.3 MB 10.1 MB/s eta 0:00:01
     ----------                               2.3/8.3 MB 10.4 MB/s eta 0:00:01
     -------------                            2.8/8.3 MB 10.5 MB/s eta 0:00:01
     ----------------                         3.3/8.3 MB 10.7 MB/s eta 0:00:01
     ------------------                       3.9/8.3 MB 10.8 MB/s eta 0:00:01
     ---------------------                    4.4/8.3 MB 10.9 MB/s eta 0:00:01
     -----------------------                  5.0/8.3 MB 10.9 MB/s eta 0:00:01
     --------------------------               5.5/8.3 MB 11.0 MB/s eta 0:00:01
     ----------------------------             6.0/8.3 MB 11.0 M


[notice] A new release of pip is available: 23.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

# 1. Initialize encoders for the agent lists
mlb_a = MultiLabelBinarizer()
mlb_b = MultiLabelBinarizer()

# 2. Transform the lists of agents into binary columns
team_a_encoded = pd.DataFrame(mlb_a.fit_transform(df_final['Team_A_Agents']), columns=['Team_A_' + col for col in mlb_a.classes_])
team_b_encoded = pd.DataFrame(mlb_b.fit_transform(df_final['Team_B_Agents']), columns=['Team_B_' + col for col in mlb_b.classes_])

# 3. Apply standard One-Hot Encoding to the Map column
map_encoded = pd.get_dummies(df_final['Map'], prefix='Map').astype(int)

# 4. Concatenate everything into the final features (X) and define target (y)
X = pd.concat([map_encoded, team_a_encoded, team_b_encoded], axis=1)
y = df_final['Target_Team_A_Win']

print("=== ENCODED FEATURES MATRIX (X) ===")
display(X.head())

=== ENCODED FEATURES MATRIX (X) ===


,Map_Abyss,Map_Ascent,Map_Bind,Map_Breeze,Map_Haven,Map_Icebox,Map_Lotus,Map_Split,Map_Sunset,Team_A_astra,...,Team_B_neon,Team_B_omen,Team_B_phoenix,Team_B_raze,Team_B_reyna,Team_B_sage,Team_B_skye,Team_B_sova,Team_B_viper,Team_B_yoru
0,0,0,0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
1,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
4,0,0,0,0,0,0,1,0,0,1,...,0,1,0,1,0,0,0,0,1,0


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# 1. Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Initialize the XGBoost classification model
# Using baseline parameters for the initial prototype
xgb_model = XGBClassifier(
    random_state=42, 
    learning_rate=0.1, 
    max_depth=5, 
    n_estimators=100,
    eval_metric='logloss'
)

# 3. Train the model using the training set
print("Training the XGBoost model...")
xgb_model.fit(X_train, y_train)

# 4. Predict the outcomes for the unseen test set
y_pred = xgb_model.predict(X_test)

# 5. Evaluate and display the model's performance
accuracy = accuracy_score(y_test, y_pred)
print(f"\n=== MODEL EVALUATION ===")
print(f"Accuracy Score: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Training the XGBoost model...

=== MODEL EVALUATION ===
Accuracy Score: 46.12%

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.48      0.47       108
           1       0.47      0.44      0.45       111

    accuracy                           0.46       219
   macro avg       0.46      0.46      0.46       219
weighted avg       0.46      0.46      0.46       219



In [18]:
import pandas as pd
import glob

# 1. Gather all file paths for scores and overviews across all years (2021-2026)
score_files = glob.glob("data/vct_*/matches/maps_scores.csv")
overview_files = glob.glob("data/vct_*/matches/overview.csv")

# 2. Read and concatenate all dataframes into massive master tables
df_scores_full = pd.concat([pd.read_csv(f) for f in score_files], ignore_index=True)
df_overview_full = pd.concat([pd.read_csv(f) for f in overview_files], ignore_index=True)

print("=== MASSIVE DATASET GATHERING ===")
print(f"Total Match Records (All Years): {len(df_scores_full)}")
print(f"Total Player Records (All Years): {len(df_overview_full)}")

C:\Users\Febrio\AppData\Local\Temp\ipykernel_33772\1330636108.py:10: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_overview_full = pd.concat([pd.read_csv(f) for f in overview_files], ignore_index=True)


=== MASSIVE DATASET GATHERING ===
Total Match Records (All Years): 27484
Total Player Records (All Years): 1149895


In [19]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# 1. Filter sides and group by the strict composite key for the massive dataset
df_agents_full = df_overview_full[df_overview_full['Side'] == 'both'].copy()
composite_keys = ['Tournament', 'Stage', 'Match Name', 'Map']

df_team_comps_full = df_agents_full.groupby(composite_keys + ['Team'])['Agents'].apply(list).reset_index()
df_team_comps_full = df_team_comps_full[df_team_comps_full['Map'] != 'All Maps']

# 2. Enforce exact 5 unique agents per team
df_team_comps_full['Agents'] = df_team_comps_full['Agents'].apply(lambda x: list(set(x)))
df_team_comps_full = df_team_comps_full[df_team_comps_full['Agents'].apply(len) == 5]

# 3. Double merge to build the final match history table
df_final_full = pd.merge(
    df_scores_full[composite_keys + ['Team A', 'Team B', 'Team A Score', 'Team B Score']], 
    df_team_comps_full, 
    left_on=composite_keys + ['Team A'], 
    right_on=composite_keys + ['Team'], 
    how='inner'
).rename(columns={'Agents': 'Team_A_Agents'}).drop(columns=['Team'])

df_final_full = pd.merge(
    df_final_full, 
    df_team_comps_full, 
    left_on=composite_keys + ['Team B'], 
    right_on=composite_keys + ['Team'], 
    how='inner'
).rename(columns={'Agents': 'Team_B_Agents'}).drop(columns=['Team'])

# 4. Generate the binary target based strictly on numerical scores
df_final_full['Target_Team_A_Win'] = (df_final_full['Team A Score'] > df_final_full['Team B Score']).astype(int)

# 5. Execute One-Hot Encoding
mlb_a = MultiLabelBinarizer()
mlb_b = MultiLabelBinarizer()
team_a_encoded = pd.DataFrame(mlb_a.fit_transform(df_final_full['Team_A_Agents']), columns=['Team_A_' + col for col in mlb_a.classes_])
team_b_encoded = pd.DataFrame(mlb_b.fit_transform(df_final_full['Team_B_Agents']), columns=['Team_B_' + col for col in mlb_b.classes_])
map_encoded = pd.get_dummies(df_final_full['Map'], prefix='Map').astype(int)

X_full = pd.concat([map_encoded, team_a_encoded, team_b_encoded], axis=1)
y_full = df_final_full['Target_Team_A_Win']

# 6. Train-Test Split and XGBoost Training on the massive dataset
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

xgb_model = XGBClassifier(random_state=42, learning_rate=0.1, max_depth=5, n_estimators=100, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
print(f"=== MASSIVE DATASET EVALUATION ===")
print(f"Total valid 5v5 matches processed: {len(df_final_full)}")
print(f"New Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

=== MASSIVE DATASET EVALUATION ===
Total valid 5v5 matches processed: 26920
New Accuracy Score: 57.06%



In [20]:
from sklearn.model_selection import GridSearchCV

# 1. Define the parameter grid to explore
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300]
}

# 2. Initialize a fresh baseline XGBoost classifier
xgb_base = XGBClassifier(random_state=42, eval_metric='logloss')

# 3. Configure GridSearchCV with 3-fold cross-validation
# n_jobs=-1 utilizes all available CPU cores to speed up processing
grid_search = GridSearchCV(
    estimator=xgb_base, 
    param_grid=param_grid, 
    scoring='accuracy', 
    cv=3, 
    verbose=1,
    n_jobs=-1
)

# 4. Execute the grid search on the training dataset
print("Starting Hyperparameter Tuning (this may take a few minutes)...")
grid_search.fit(X_train, y_train)

# 5. Extract the optimized model and evaluate its new accuracy
best_xgb = grid_search.best_estimator_
y_pred_tuned = best_xgb.predict(X_test)

print(f"\n=== HYPERPARAMETER TUNING RESULTS ===")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Final Tuned Accuracy: {accuracy_score(y_test, y_pred_tuned) * 100:.2f}%")

Starting Hyperparameter Tuning (this may take a few minutes)...
Fitting 3 folds for each of 27 candidates, totalling 81 fits


KeyboardInterrupt: 

In [21]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# 1. Define the parameter grid to explore
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300]
}

# 2. Initialize XGBoost with GPU acceleration enabled
# 'gpu_hist' is the standard hardware acceleration flag for XGBoost 1.6.x
xgb_base = XGBClassifier(
    random_state=42, 
    eval_metric='logloss',
    tree_method='gpu_hist'
)

# 3. Configure GridSearchCV without the CPU override to prevent Windows deadlock
grid_search = GridSearchCV(
    estimator=xgb_base, 
    param_grid=param_grid, 
    scoring='accuracy', 
    cv=3, 
    verbose=1
)

# 4. Execute the grid search on the training dataset
print("Starting GPU-accelerated Hyperparameter Tuning...")
grid_search.fit(X_train, y_train)

# 5. Extract the optimized model and evaluate its new accuracy
best_xgb = grid_search.best_estimator_
y_pred_tuned = best_xgb.predict(X_test)

print("\n=== HYPERPARAMETER TUNING RESULTS ===")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Final Tuned Accuracy: {accuracy_score(y_test, y_pred_tuned) * 100:.2f}%")

Starting GPU-accelerated Hyperparameter Tuning...
Fitting 3 folds for each of 27 candidates, totalling 81 fits


ValueError: 
All the 81 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
81 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Febrio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Febrio\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "c:\Users\Febrio\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\sklearn.py", line 1806, in fit
    self._Booster = train(
                    ^^^^^^
  File "c:\Users\Febrio\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py"

In [22]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# 1. Define the parameter grid to explore
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300]
}

# 2. Initialize XGBoost using the modern histogram method and CUDA GPU device
xgb_base = XGBClassifier(
    random_state=42, 
    eval_metric='logloss',
    tree_method='hist',
    device='cuda'
)

# 3. Configure GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_base, 
    param_grid=param_grid, 
    scoring='accuracy', 
    cv=3, 
    verbose=1
)

# 4. Execute the grid search on the training dataset
print("Starting Optimized Hyperparameter Tuning...")
grid_search.fit(X_train, y_train)

# 5. Extract the optimized model and evaluate its new accuracy
best_xgb = grid_search.best_estimator_
y_pred_tuned = best_xgb.predict(X_test)

print("\n=== HYPERPARAMETER TUNING RESULTS ===")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Final Tuned Accuracy: {accuracy_score(y_test, y_pred_tuned) * 100:.2f}%")


Starting Optimized Hyperparameter Tuning...
Fitting 3 folds for each of 27 candidates, totalling 81 fits


c:\Users\Febrio\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:751: UserWarning: [11:45:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



=== HYPERPARAMETER TUNING RESULTS ===
Best Parameters Found: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300}
Final Tuned Accuracy: 57.71%


In [23]:
import joblib
import os

# 1. Create a dedicated directory to store the exported model files
os.makedirs('saved_models', exist_ok=True)

# 2. Export the optimized XGBoost model (the machine learning brain)
joblib.dump(best_xgb, 'saved_models/xgboost_vct_model.pkl')

# 3. Export the MultiLabelBinarizers and Map columns (the data translators)
# This is mandatory so the web app knows how to format new text inputs into the exact 57-column matrix
encoding_assets = {
    'mlb_a': mlb_a,
    'mlb_b': mlb_b,
    'map_columns': map_encoded.columns
}
joblib.dump(encoding_assets, 'saved_models/encoding_assets.pkl')

print("=== MODEL EXPORT SUCCESSFUL ===")
print("Saved files in the 'saved_models' directory:")
print("1. xgboost_vct_model.pkl")
print("2. encoding_assets.pkl")

=== MODEL EXPORT SUCCESSFUL ===
Saved files in the 'saved_models' directory:
1. xgboost_vct_model.pkl
2. encoding_assets.pkl


In [24]:
import pandas as pd
import numpy as np

# 1. Define your custom match scenario for manual inference
custom_map = "Ascent"
team_a_picks = ["jett", "omen", "kayo", "killjoy", "sova"]
team_b_picks = ["neon", "astra", "fade", "cypher", "breach"]

# 2. Transform the text inputs into the exact binary matrix format
# We strictly use mlb.transform (not fit_transform) to apply the existing training rules
team_a_custom = pd.DataFrame(mlb_a.transform([team_a_picks]), columns=['Team_A_' + col for col in mlb_a.classes_])
team_b_custom = pd.DataFrame(mlb_b.transform([team_b_picks]), columns=['Team_B_' + col for col in mlb_b.classes_])

# 3. Create a zero-filled dataframe for the maps, then set the chosen map to 1
map_custom = pd.DataFrame(0, index=[0], columns=map_encoded.columns)
map_column_name = f"Map_{custom_map}"
if map_column_name in map_custom.columns:
    map_custom[map_column_name] = 1

# 4. Concatenate all features into a single row mimicking the training structure
X_custom = pd.concat([map_custom, team_a_custom, team_b_custom], axis=1)

# 5. Predict the win probabilities using predict_proba()
# Index 0 is the probability of class 0 (Team B wins), Index 1 is class 1 (Team A wins)
probabilities = best_xgb.predict_proba(X_custom)[0]
team_b_win_prob = probabilities[0] * 100
team_a_win_prob = probabilities[1] * 100

print("=== CUSTOM MATCH PREDICTION ===")
print(f"Map Played: {custom_map}")
print(f"Team A Draft: {team_a_picks}")
print(f"Team B Draft: {team_b_picks}\n")
print("=== WIN PROBABILITY RESULT ===")
print(f"Team A Win Chance: {team_a_win_prob:.2f}%")
print(f"Team B Win Chance: {team_b_win_prob:.2f}%")

=== CUSTOM MATCH PREDICTION ===
Map Played: Ascent
Team A Draft: ['jett', 'omen', 'kayo', 'killjoy', 'sova']
Team B Draft: ['neon', 'astra', 'fade', 'cypher', 'breach']

=== WIN PROBABILITY RESULT ===
Team A Win Chance: 62.60%
Team B Win Chance: 37.40%
